# Exploratory Data Analysis (EDA) - Fake Review Detector

This notebook explores the preprocessed fake review dataset to understand class distributions, review length profiles, formatting characteristics (like capital letter usage and punctuation), and textual features that differentiate genuine reviews from fake reviews.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 1. Load Data
We load the preprocessed data containing both the raw text, cleaned text, and extracted structural features.

In [ ]:
processed_data_path = Path("../data/processed/fake_reviews_processed.parquet")

if not processed_data_path.exists():
    # Fallback to local running directory if executed from repo root
    processed_data_path = Path("data/processed/fake_reviews_processed.parquet")

if not processed_data_path.exists():
    raise FileNotFoundError(f"Processed dataset not found at {processed_data_path}. Please run 'uv run python -m src.pipeline' first.")

df = pd.read_parquet(processed_data_path)
print(f"Dataset successfully loaded!")
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

## 2. Target Class Distribution
Let's see if the dataset is balanced between Genuine (0) and Fake (1) reviews.

In [ ]:
class_counts = df["label"].value_counts()
class_percentages = df["label"].value_counts(normalize=True) * 100

print("Class Counts:")
for label, count in class_counts.items():
    name = "Fake (1)" if label == 1 else "Genuine (0)"
    print(f"  {name}: {count:,} ({class_percentages[label]:.2f}%)")

plt.figure(figsize=(6, 5))
ax = sns.barplot(x=class_counts.index.map({0: "Genuine", 1: "Fake"}), y=class_counts.values, hue=class_counts.index, palette="coolwarm", legend=False)
plt.title("Distribution of Review Classes", fontsize=14, fontweight="bold")
plt.xlabel("Class")
plt.ylabel("Count")
for p in ax.patches:
    ax.annotate(f'{p.get_height():,.0f}', (p.get_x() + p.get_width() / 2., p.get_height() * 0.9),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Structural Features Analysis
We compare the distributions of review length, average word length, capital ratio, and punctuation ratios for genuine vs fake reviews. 

In [ ]:
# Compute average values for features grouped by label
feature_cols = ["char_count", "word_count", "avg_word_length", "cap_ratio", "exclamation_ratio", "question_ratio"]
grouped_stats = df.groupby("label")[feature_cols].mean().round(4)
grouped_stats.index = grouped_stats.index.map({0: "Genuine", 1: "Fake"})
print("Average feature values by class:")
grouped_stats

### 3.1 Review Length (Word Count)
Are fake reviews typically shorter or longer than genuine reviews?

In [ ]:
plt.figure(figsize=(12, 5))

# Histogram/Density Plot (capped at 99th percentile to exclude extreme outliers for better visualization)
plt.subplot(1, 2, 1)
limit = df["word_count"].quantile(0.99)
sns.histplot(data=df[df["word_count"] <= limit], x="word_count", hue="label", kde=True, element="step", stat="density", common_norm=False, palette="coolwarm")
plt.title("Review Word Count Distribution (99th percentile)")
plt.xlabel("Word Count")

# Box Plot
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="label", y="word_count", palette="coolwarm", hue="label", legend=False)
plt.title("Review Word Count Boxplot (Full Range)")
plt.xlabel("Class (0=Genuine, 1=Fake)")
plt.ylabel("Word Count")
plt.yscale("log")  # Log scale to handle outliers gracefully

plt.tight_layout()
plt.show()

### 3.2 Uppercase Ratio (cap_ratio)
Do fake reviews feature more capital letters (e.g. dramatic "shouting" style or bad formatting)?

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
limit_cap = df["cap_ratio"].quantile(0.99)
sns.kdeplot(data=df[df["cap_ratio"] <= limit_cap], x="cap_ratio", hue="label", fill=True, common_norm=False, palette="coolwarm")
plt.title("Uppercase Character Ratio Density")
plt.xlabel("Capital Ratio (Capital Letters / Total Chars)")

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="label", y="cap_ratio", palette="coolwarm", hue="label", legend=False)
plt.title("Uppercase Character Ratio Boxplot")
plt.xlabel("Class (0=Genuine, 1=Fake)")
plt.ylabel("Capital Ratio")

plt.tight_layout()
plt.show()

### 3.3 Exclamation and Punctuation Intensity
Let's see if fake reviews employ more intense punctuation (like exclamation marks) to convey artificial sentiment.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
limit_excl = df["exclamation_ratio"].quantile(0.99)
sns.kdeplot(data=df[df["exclamation_ratio"] <= limit_excl], x="exclamation_ratio", hue="label", fill=True, common_norm=False, palette="coolwarm")
plt.title("Exclamation Mark Ratio Density")
plt.xlabel("Exclamation Ratio")

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="label", y="exclamation_ratio", palette="coolwarm", hue="label", legend=False)
plt.title("Exclamation Mark Ratio Boxplot")
plt.xlabel("Class (0=Genuine, 1=Fake)")
plt.ylabel("Exclamation Ratio")

plt.tight_layout()
plt.show()

## 4. Text Content & Vocabulary Analysis
Let's analyze the vocabulary differences. We look at the most frequent unigrams (single words) in both genuine and fake reviews, excluding basic english stop words.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_n_words(corpus, n=15):
    vec = CountVectorizer(stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return pd.DataFrame(words_freq[:n], columns=['Word', 'Frequency'])

print("Extracting top words for Genuine reviews...")
genuine_top = get_top_n_words(df[df['label'] == 0]['clean_text'], n=15)

print("Extracting top words for Fake reviews...")
fake_top = get_top_n_words(df[df['label'] == 1]['clean_text'], n=15)

plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
sns.barplot(data=genuine_top, x='Frequency', y='Word', color='skyblue')
plt.title('Top 15 Words in Genuine Reviews')
plt.xlabel('Frequency')

plt.subplot(1, 2, 2)
sns.barplot(data=fake_top, x='Frequency', y='Word', color='salmon')
plt.title('Top 15 Words in Fake (Computer-Generated) Reviews')
plt.xlabel('Frequency')

plt.tight_layout()
plt.show()

## 5. Key Findings for Modeling

Based on the analysis, here are the main insights to apply when training models:

1. **Class Balance**: The dataset is highly balanced, meaning metrics like Accuracy and F1-score are suitable targets.
2. **Review Length**: Examine whether fake/deceptive reviews exhibit different length signatures compared to human reviews. Computer-generated reviews might have highly consistent lengths or smaller variance.
3. **Style & Capitalization**: Check if capitalization styles (`cap_ratio`) differ significantly, showing stylistic anomalies in how text is generated/formatted.
4. **Punctuation Intensity**: Check for differences in emotional markers like exclamations.
5. **Vocabulary Divergence**: Vocabulary distributions can reveal distinct phrases or templates used by generators compared to normal, informal consumer reviews.